# World Cup 2026 — Match & Tournament Prediction (results-based)

Predict 2026 World Cup match outcomes and tournament-level probabilities from **international
match results only** (goals + fixtures) — no xG or player-level data. This deliberately trades
feature richness for sample size, avoiding the ~280-match data ceiling that constrained the
previous StatsBomb-based project.

The data is `martj42/international_results`: **49,496** internationals from 1872 to present
(`date, home_team, away_team, home_score, away_score, tournament, city, country, neutral`), plus
`shootouts.csv` (for knockout resolution) and `goalscorers.csv` (minute-level, optional). Because
the raw columns are minimal, almost every useful feature is **derived** — team-strength ratings,
recent form, tournament importance, venue context — all computed strictly as-of the match date to
avoid lookahead.

## Modeling plan
- **Poisson / Dixon-Coles goals model is the spine.** The 2026 format resolves on **goal
  difference** (group-stage tiebreakers and picking the 8 best third-placed teams), and the Monte
  Carlo simulator samples scorelines — so we need goals, not just W/D/L. Win/draw/loss
  probabilities fall out of the Poisson PMF for free. Dixon-Coles adds the low-score correlation
  correction and time-weighting.
- **Then a direct W/D/L classifier** on the same features, tested against the PMF-derived
  probabilities on time-ordered holdout — sequenced after the spine, not built in parallel. (This
  was rejected at 280 matches last project; with ~49k it gets a fair re-test.)
- **Monte Carlo tournament simulator** driven by the combined λ's.
- **Evaluation:** RPS, log loss, Brier — calibration-first (the simulator compounds per-match
  error across rounds). **Walk-forward validation only** (no random K-fold — leakage). Baselines
  to beat: Elo and bookmaker closing odds.

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
results = pd.read_csv('../data/raw/results.csv')
goalscorers = pd.read_csv('../data/raw/goalscorers.csv')
shootouts = pd.read_csv('../data/raw/shootouts.csv')

print(results.info())
print(results.shape)
print('\n')
print(goalscorers.info())
print(goalscorers.shape)
print('\n')
print(shootouts.info())
print(shootouts.shape)

<class 'pandas.DataFrame'>
RangeIndex: 49496 entries, 0 to 49495
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49496 non-null  str    
 1   home_team   49496 non-null  str    
 2   away_team   49496 non-null  str    
 3   home_score  49484 non-null  float64
 4   away_score  49484 non-null  float64
 5   tournament  49496 non-null  str    
 6   city        49496 non-null  str    
 7   country     49496 non-null  str    
 8   neutral     49496 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB
None
(49496, 9)


<class 'pandas.DataFrame'>
RangeIndex: 47837 entries, 0 to 47836
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   date       47837 non-null  str    
 1   home_team  47837 non-null  str    
 2   away_team  47837 non-null  str    
 3   team       47837 non-null  str    
 4   scorer     47793 non-null  str    


### What the initial load shows

- **49,496 matches**, 9 columns. `neutral` parses cleanly as `bool` (no misalignment despite
  quoted city names like `"Washington, D.C."`).
- **12 matches have missing scores** (`home_score`/`away_score` = 49,484 non-null) — likely
  unplayed or abandoned fixtures; drop or handle before modeling.
- `goalscorers` and `shootouts` load fine; `shootouts.first_shooter` is mostly null (expected —
  rarely recorded).

Distribution checks (below) surface the decisions that shape data scope:

- **Friendlies dominate** — ~37% of all matches (18,388). Low-stakes, rotated squads, often
  lopsided → candidates for down-weighting rather than equal weight.
- **Qualifier-heavy**: World Cup / Euro / AFCON qualifiers make up the bulk; actual tournament
  *finals* are a small slice, and many qualifiers are giant-vs-minnow blowouts.
- **Front-loaded to the modern era**: ~32k matches are from 1990 on, ~4k before 1950. Once
  recency-weighted, the pre-war era contributes almost nothing — the effective sample is modern.
- **200 distinct tournaments** → need an importance *tiering*, not 200 dummy variables.
- Domain prior: the same ~6 nations win almost every World Cup (Brazil, Argentina, Germany,
  Italy, England, France) → the team-strength rating carries most of the signal; the model's job
  is to rank favorites and stay calibrated, not to pick upsets.